# Meta Motivo — swap-XML tutorial

Adapted from the [metamotivo](https://github.com/facebookresearch/metamotivo) repo's own
`tutorial.ipynb`, changed only to (a) point `make_humenv` at an MJCF file from this project's
`mjcf/` folder instead of HumEnv's bundled rig, and (b) add a compatibility preflight check.

**Read this before swapping in a different XML.** Meta Motivo's actor/critic networks have a
*fixed* input (observation) and output (action) size, set by the skeleton it was trained on:
HumEnv's own `humenv/assets/robot.xml` (SMPL body names — `Pelvis`, `L_Hip`, ... `R_Hand` — 24
bodies, 69 actuators). `HumEnv(xml=...)` happily accepts any MJCF, but the pretrained model has
no idea what to do with a different skeleton — if the new XML's actuator count or joint
ordering differs from that rig, `model.act()` fails on a shape mismatch (or silently drives the
wrong joints if the counts coincidentally match).

`mjcf/humenv_robot.xml` in this repo is an **unmodified copy** of that exact rig (CC BY-NC 4.0,
from the HumEnv repo), so it loads and runs as-is — `XML_PATH` below defaults to it. To test a
different body architecture, make a scaled *copy* of `humenv_robot.xml` (same body/joint/actuator
names and gear ratios, only different bone lengths / geom sizes / mass — the same kind of scaling
this repo already applied to derive `humanoid_stocky.xml`/`humanoid_tall.xml` from
`humanoid_CMU.xml`) and point `XML_PATH` at the copy. This project's `mjcf/humanoid_CMU.xml`
itself is **not** usable here — it's a different, 57-actuator dm_control-style rig with different
joint names, incompatible with the pretrained model's fixed input/output size; the preflight cell
below will catch that if you try.


## All imports

In [1]:
import os

# Headless GPU rendering: no X server / DISPLAY is available on this machine, so MuJoCo's
# default GLX backend can't create an OpenGL context (-> "OpenGL platform library has not
# been loaded" FatalError from mjr_makeContext, which aborts the whole process/kernel rather
# than raising a catchable Python exception). EGL renders off-screen directly against the
# NVIDIA driver without needing a display. Must be set before mujoco is imported anywhere
# below (including transitively via humenv/gymnasium), so this cell has to run first.
os.environ["MUJOCO_GL"] = "egl"


In [2]:
from packaging.version import Version
from metamotivo.fb_cpr.huggingface import FBcprModel
from huggingface_hub import hf_hub_download
from humenv import make_humenv
from humenv.env import make_from_name
import gymnasium
from gymnasium.wrappers import FlattenObservation, TransformObservation
from metamotivo.buffers.buffers import DictBuffer
from metamotivo.wrappers.humenvbench import relabel

import torch
import mediapy as media
import h5py


/home/allen19/metamotivo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Config

`XML_PATH` is the only thing you need to change to test a different body. Point it at one of
this repo's `mjcf/*.xml` files (must satisfy the compatibility preflight below), or leave it
`None` to use HumEnv's own default rig as a sanity check first.

`CAMERA` must name a `<camera>` that actually exists in `XML_PATH` — HumEnv's default rig has
`front_side`; this repo's `mjcf/*.xml` files have `back`/`side`/`egocentric` instead.

In [3]:
MODEL_NAME = "metamotivo-M-1"   # or metamotivo-S-1 .. metamotivo-S-5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

XML_PATH = "../assets/robots/robot.xml"  # swap for a scaled variant of this file, see below
CAMERA = "front_side" # match a <camera name="..."> that exists in XML_PATH
TASK = "move-ego-0-2" # see humenv.STANDARD_TASKS for the full list


## Model download

In [4]:
model = FBcprModel.from_pretrained(f"facebook/{MODEL_NAME}")
model.to(DEVICE)
print(model)


FBcprModel(
  (_backward_map): BackwardMap(
    (net): Sequential(
      (0): Linear(in_features=358, out_features=256, bias=True)
      (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
      (2): Tanh()
      (3): Linear(in_features=256, out_features=256, bias=True)
      (4): Norm()
    )
  )
  (_forward_map): ResidualForwardMap(
    (embed_z): Sequential(
      (0): Block(
        (mlp): Sequential(
          (0): ParallelLayerNorm([614], eps=1e-05, elementwise_affine=True)
          (1): DenseParallel(in_features=614, out_features=2048, n_parallel=2, bias=True)
          (2): Mish()
        )
      )
      (1): Block(
        (mlp): Sequential(
          (0): ParallelLayerNorm([2048], eps=1e-05, elementwise_affine=True)
          (1): DenseParallel(in_features=2048, out_features=1024, n_parallel=2, bias=True)
          (2): Mish()
        )
      )
    )
    (embed_sa): Sequential(
      (0): Block(
        (mlp): Sequential(
          (0): ParallelLayerNorm([4

## Build the environment (with the swapped XML)

Same wrapper stack as the original tutorial — `FlattenObservation` + `TransformObservation` so
`model.act` can consume the observation directly as a batched tensor.

In [5]:
if Version("0.26") <= Version(gymnasium.__version__) < Version("1.0"):
    transform_obs_wrapper = lambda env: TransformObservation(
        env, lambda obs: torch.tensor(obs.reshape(1, -1), dtype=torch.float32, device=DEVICE)
    )
else:
    transform_obs_wrapper = lambda env: TransformObservation(
        env, lambda obs: torch.tensor(obs.reshape(1, -1), dtype=torch.float32, device=DEVICE), env.observation_space
    )


def build_env(xml=None, camera="front_side"):
    kwargs = {"camera": camera}
    if xml is not None:
        kwargs["xml"] = xml
    return make_humenv(
        num_envs=1,
        wrappers=[FlattenObservation, transform_obs_wrapper],
        state_init="Default",
        **kwargs,
    )


env, _ = build_env(xml=XML_PATH, camera=CAMERA)


### Compatibility preflight

Compares this XML's observation/action dimensions against HumEnv's default rig (what the
pretrained model actually expects). Raises immediately with a clear message instead of failing
deep inside a matrix multiply.

In [6]:
if XML_PATH is None:
    print("XML_PATH is None (using HumEnv's default rig) -- nothing to check.")
else:
    reference_env, _ = build_env(xml=None)
    expected_obs = reference_env.observation_space.shape[0]
    expected_actions = reference_env.action_space.shape[0]
    reference_env.close()

    actual_obs = env.observation_space.shape[0]
    actual_actions = env.action_space.shape[0]

    print(f"expected (HumEnv default rig) -> obs: {expected_obs}, actions: {expected_actions}")
    print(f"XML_PATH ({XML_PATH})         -> obs: {actual_obs}, actions: {actual_actions}")

    if (actual_obs, actual_actions) != (expected_obs, expected_actions):
        raise ValueError(
            "This XML's observation/action dimensions don't match what the pretrained Motivo "
            "model expects. It must keep HumEnv's default rig's body/joint/actuator names and "
            "gear ratios, only varying morphology -- see the markdown note at the top."
        )
    print("OK: dimensions match, the pretrained model can drive this rig.")


expected (HumEnv default rig) -> obs: 358, actions: 69
XML_PATH (robot.xml)         -> obs: 358, actions: 69
OK: dimensions match, the pretrained model can drive this rig.


## Run a policy

Sample a random context `z` and roll out the (untargeted) policy — same as the original
tutorial's first rollout, just against whichever rig `env` was built with above.

In [7]:
z = model.sample_z(1, device=DEVICE)
observation, _ = env.reset()
frames = [env.render()]
for _ in range(30):
    action = model.act(observation, z, mean=True)
    observation, reward, terminated, truncated, info = env.step(action.cpu().numpy().ravel())
    frames.append(env.render())

media.show_video(frames, fps=30)


libEGL warning: failed to open /dev/dri/renderD129: Permission denied

libEGL warning: failed to open /dev/dri/renderD129: Permission denied

libEGL warning: failed to open /dev/dri/card2: Permission denied

libEGL warning: failed to open /dev/dri/renderD128: Permission denied

libEGL warning: failed to open /dev/dri/renderD128: Permission denied

libEGL warning: failed to open /dev/dri/card1: Permission denied



This browser does not support the video tag.


## Reward-prompted rollout

Downloads the model's reward-inference buffer, labels it with a named HumEnv task, infers a
context `z` for that task, then rolls out the resulting policy — same flow as the original
tutorial's reward-prompting section.

In [8]:
local_dir = f"../data/{MODEL_NAME}-datasets"
buffer_path = hf_hub_download(
    repo_id=f"facebook/{MODEL_NAME}",
    filename="data/buffer_inference_500000.hdf5",
    repo_type="model",
    local_dir=local_dir,
)

with h5py.File(buffer_path, "r") as hf:
    data = {k: v[:] for k, v in hf.items()}
buffer = DictBuffer(capacity=data["qpos"].shape[0], device="cpu")
buffer.extend(data)
del data


In [9]:
reward_fn = make_from_name(TASK)

N = 100_000
batch = buffer.sample(N)
rewards = relabel(
    env,
    qpos=batch["next_qpos"],
    qvel=batch["next_qvel"],
    action=batch["action"],
    reward_fn=reward_fn,
    max_workers=8,
)

z = model.reward_wr_inference(
    next_obs=batch["next_observation"],
    reward=torch.tensor(rewards, device=DEVICE, dtype=torch.float32),
)
print(f"z shape: {z.shape}")


z shape: torch.Size([1, 256])


In [10]:
observation, _ = env.reset()
frames = [env.render()]
for _ in range(30):
    action = model.act(observation, z, mean=True)
    observation, reward, terminated, truncated, info = env.step(action.cpu().numpy().ravel())
    frames.append(env.render())

media.show_video(frames, fps=30)


This browser does not support the video tag.


## Quantify rollout quality (task return)

The two rollouts above only give you a video — `env`'s task reward is never set (defaults to
`ZeroReward`), so the `reward` values `env.step()` returned in those loops were always 0. This
section temporarily switches the env's task to `TASK`'s reward function (the same `reward_fn`
used above to relabel the buffer and infer `z`) and reports the cumulative return each policy
actually earns on it — a number you can compare across XML swaps, not just eyeball a video for.


In [12]:
def rollout_return(z, steps=30):
    env.unwrapped.set_task(reward_fn)
    try:
        observation, _ = env.reset()
        total_reward = 0.0
        for _ in range(steps):
            action = model.act(observation, z, mean=True)
            observation, reward, terminated, truncated, info = env.step(action.cpu().numpy().ravel())
            total_reward += float(reward)
    finally:
        env.unwrapped.set_task(None)
    return total_reward


z_random = model.sample_z(1, device=DEVICE)
return_random = rollout_return(z_random)
return_prompted = rollout_return(z)

print(f"TASK = {TASK!r}")
print(f"untargeted (random z) return over 30 steps: {return_random:.3f}")
print(f"reward-prompted z return over 30 steps: {return_prompted:.3f}")

TASK = 'move-ego-0-2'
untargeted (random z) return over 30 steps: 7.174
reward-prompted z return over 30 steps: 22.890
